# Даалгавар 2–4: Өгөгдлийн дүрслэл

Энэ notebook дотор даалгавар бүрийн өгөгдөл бэлтгэл, дүрслэл, ажиглалт болон тайлбарыг шууд гүйцэтгэнэ. Тусдаа PDF тайлан шаардахгүй.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE = Path.cwd()
sns.set_theme(style='whitegrid', context='notebook')
weather = pd.read_csv(BASE / 'weatherHistory.csv')
retail = pd.read_csv(BASE / 'retail_sales_dataset.csv', parse_dates=['Date'])
adult = pd.read_csv(BASE / 'adult.csv')
print('Weather:', weather.shape)
print('Retail:', retail.shape)
print('Adult:', adult.shape)


## Даалгавар 2 — Weather History

Эх өгөгдөл цаг тутмын давтамжтай тул нэг өдөрт агрегатална. Эх өгөгдөлд үүлэрхэг байдлын 0–8 тоон багана байхгүй учраас `Summary` баганыг үүлэрхэг байдлын төлөөлөл болгон хадгална.

In [ ]:
weather['Formatted Date'] = pd.to_datetime(weather['Formatted Date'], utc=True)
def get_season(month):
    if month in [12, 1, 2]: return 'Өвөл'
    if month in [3, 4, 5]: return 'Хавар'
    if month in [6, 7, 8]: return 'Зун'
    return 'Намар'

weather['date'] = weather['Formatted Date'].dt.date
weather['season'] = weather['Formatted Date'].dt.month.map(get_season)
daily_weather = weather.groupby(['date', 'season'], as_index=False).agg(
    temperature_c=('Temperature (C)', 'mean'),
    humidity=('Humidity', 'mean'),
    rainy_hours=('Precip Type', lambda x: (x == 'rain').sum()),
    cloud_summary=('Summary', lambda x: x.mode().iat[0])
)
daily_weather['date'] = pd.to_datetime(daily_weather['date'])
daily_weather.head()

### Chi (2000) загварын төлөвүүд

- **Өгөгдлийн үнэ цэнэ:** `weatherHistory.csv` түүхий өгөгдөл
- **Шинжилгээний ангилал:** өдөр болон улирлаар бүлэглэсэн хүснэгт
- **Дүрслэлийн ангилал:** хамаарал, хугацааны цуваа, тархалт
- **Зурган өгөгдөл:** доорх эцсийн графикууд

In [ ]:
season_colors = {'Өвөл': '#4B9CD3', 'Хавар': '#6ABF69', 'Зун': '#E87722', 'Намар': '#9467BD'}
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for name, group in daily_weather.groupby('season'):
    axes[0].scatter(group['temperature_c'], group['humidity'], s=18, alpha=.55,
                    color=season_colors[name], label=name)
axes[0].set_title('Scatter: Температур vs Чийгшил')
axes[0].set_xlabel('Температур (°C)')
axes[0].set_ylabel('Чийгшил (0–1)')
axes[0].legend(title='Улирал')

axes[1].plot(daily_weather['date'], daily_weather['temperature_c'], color='#C0392B', linewidth=1)
axes[1].set_title('Line: Температурын хугацааны цуваа')
axes[1].set_xlabel('Огноо')
axes[1].set_ylabel('Температур (°C)')
axes[1].tick_params(axis='x', rotation=35)

axes[2].hist(daily_weather['rainy_hours'], bins=np.arange(-.5, 25.5, 1),
             color='#2E86AB', edgecolor='white')
axes[2].set_title('Histogram: Хур тунадасны тархалт')
axes[2].set_xlabel('Бороо орсон цагийн тоо / өдөр')
axes[2].set_ylabel('Өдрийн тоо')

fig.suptitle('Даалгавар 2 — Weather History', fontsize=16, weight='bold')
plt.tight_layout()
plt.show()

### Weather-ийн олдвор ба аномали

- Зун температур өндөр, өвөл бага; улирлаас хамаарсан хэв маяг тодорхой харагдана.
- Температур огцом өөрчлөгдсөн өдөр болон бороо олон цаг үргэлжилсэн өдөр нь шалгах аномалийн нэр дэвшигч.
- Боломжит шалтгаан: цаг агаарын фронт, шуурга, эсвэл хэмжилтийн алдаа.
- Дараагийн шинжилгээ: rolling mean, улирал тус бүрийн IQR/z-score, missing/duplicate шалгалт, цагийн түүхий мөрөөр буцааж нягтлах.

## Даалгавар 3 — Retail Sales

`retail_sales_dataset.csv` нь салбарын баганагүй тул `Product Category`-гаар бодит өгөгдлийг шинжилнэ. Даалгаварт өгсөн 5 салбарын хүснэгтийг муу/сайн дүрслэлийн шалгуурт тусад нь ашиглав.

In [ ]:
category_sales = retail.groupby('Product Category', as_index=False)['Total Amount'].sum()
plt.figure(figsize=(8, 4))
sns.barplot(data=category_sales, x='Product Category', y='Total Amount',
            hue='Product Category', palette='Set2', legend=False)
plt.title('Retail Sales: ангиллаар нийт борлуулалт')
plt.xlabel('Бүтээгдэхүүний ангилал')
plt.ylabel('Нийт дүн')
plt.show()
category_sales

In [ ]:
branches = pd.DataFrame({'Салбар': list('АБВГД'), 'Q1': [95,45,12,78,66],
                         'Q2': [98,67,34,82,72], 'Q3': [97,89,56,88,81],
                         'Q4': [99,23,78,91,93]})
long = branches.melt('Салбар', var_name='Улирал', value_name='Борлуулалт')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(long['Салбар'] + '-' + long['Улирал'], long['Борлуулалт'], color='green')
axes[0].set_title('Муу дүрслэл')
axes[0].set_ylabel('Борлуулалт')
axes[0].set_ylim(90, 100)
axes[0].tick_params(axis='x', rotation=70)

for branch, group in long.groupby('Салбар'):
    axes[1].plot(group['Улирал'], group['Борлуулалт'], marker='o', linewidth=2, label=f'Салбар {branch}')
axes[1].set_title('Сайн дүрслэл: шударга масштаб ба ялгаатай тэмдэглэгээ')
axes[1].set_xlabel('Улирал')
axes[1].set_ylabel('Борлуулалтын индекс')
axes[1].set_ylim(0, 105)
axes[1].legend(ncol=2)
plt.tight_layout()
plt.show()

### 4 шалгуураар үнэлэх

1. **Шударга:** 90–100 хүртэл тасалсан y тэнхлэг нь бага борлуулалтыг нууж, ялгааг хэтрүүлнэ. Сайн хувилбар 0–105 масштабтай.
2. **Ойлгомжтой:** “График” гэх ерөнхий гарчиг, бүгд ногоон өнгө нь мэдээллийг ялгахгүй. Сайн хувилбар тодорхой гарчиг, legend-тэй.
3. **Хялбар:** Салбар-улирал бүрийг тусдаа баганаар харуулахад шошго давчуу. Сайн хувилбар улирлыг x тэнхлэгт, салбар бүрийг нэг шугамаар харуулсан.
4. **Үнэ цэнэтэй:** Сайн график аль салбар улирлаар өсөж/буурч байгааг шууд харуулна.

## Даалгавар 4 — Adult dataset: 3 гогцооны процесс

In [ ]:
adult['income_label'] = adult['income'].str.replace('.', '', regex=False).str.strip()
adult['age_group'] = pd.cut(adult['age'], [0, 25, 35, 45, 55, 65, 100],
                           labels=['≤25', '26–35', '36–45', '46–55', '56–65', '66+'])
age_income = adult.groupby('age_group', observed=False)['income_label'].apply(lambda x: (x == '>50K').mean() * 100)
education_income = adult.groupby('education')['income_label'].apply(lambda x: (x == '>50K').mean() * 100).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
age_income.plot(kind='bar', color='#3B82A0', ax=axes[0])
axes[0].set_title('Хайгуул: насны бүлгээр >50K орлогын хувь')
axes[0].set_xlabel('Насны бүлэг')
axes[0].set_ylabel('Хувь (%)')
axes[0].tick_params(axis='x', rotation=0)
education_income.sort_values().plot(kind='barh', color='#D97A2B', ax=axes[1])
axes[1].set_title('Хайгуул: боловсролоор >50K орлогын хувь')
axes[1].set_xlabel('Хувь (%)')
axes[1].set_ylabel('Боловсрол')
plt.tight_layout()
plt.show()


### Хайгуулын гогцоо

Насны бүлэг, хүйс, боловсрол, орлогын тархалтыг bar chart, histogram, boxplot, stacked bar-аар хайна. Анхны ажиглалт: нас болон боловсролын бүлгүүдэд `>50K` орлогын хувь ялгаатай байна.

### Баталгаажуулалтын гогцоо

Таамаглал: боловсролын түвшин болон долоо хоногт ажилласан цаг нь өндөр орлоготой холбоотой. Үүнийг `?` missing утгыг цэвэрлэсний дараа chi-square, confidence interval, logistic regression болон confounder шалгалтаар баталгаажуулна. Графикийн холбоог шууд шалтгаан гэж тайлбарлахгүй.

### Мэдлэг үүсгэх гогцоо

Эцсийн дүгнэлтийг өгөгдлийн хязгаарлалттай нь хамт бичнэ. Хөдөлмөрийн зах зээлийн мэргэжилтэнтэй үр дүнг шалгаж, сургалт/ажлын байрны бодлогод шударга, тайлбарлагдах байдлаар ашиглана.